# Vectors and feature vectors

Code for the Note `note.md`: vectors in NumPy, iris feature vectors, encoding a Titanic passenger, and a tiny bag-of-words recommender.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.feature_extraction.text import CountVectorizer

## 1. A vector is a list of numbers
The point (3, 4) in 2D, and a vector in 5 dimensions.

In [ ]:
a = np.array([3, 4])
print(a, 'dimension:', a.size)

# an n-dimensional vector works the same way
x = np.array([1.5, -2, 0, 7, 3])
print(x, 'dimension:', x.size)

## 2. Iris: every flower is a feature vector

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data          # the four input columns
y = iris.target        # the target column (0, 1, 2 = species)
print(X.shape)         # 150 flowers, 4 inputs each

# the first flower's feature vector
first = X.iloc[0].to_numpy()
print(first, iris.target_names[y[0]])

In [ ]:
# Do flowers of the same species sit close together?
# Predict each flower's species from its nearest other flower (leave-one-out 1-NN),
# then repeat with the species shuffled, which breaks any link between closeness and species.
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1)
acc = cross_val_score(knn, X, y, cv=LeaveOneOut()).mean()
y_shuffled = y.sample(frac=1, random_state=0).reset_index(drop=True)
acc_shuffled = cross_val_score(knn, X, y_shuffled, cv=LeaveOneOut()).mean()
print(f"nearest-flower accuracy: {acc:.3f}; with shuffled species: {acc_shuffled:.3f}")

## 3. Text values must be encoded
A Titanic passenger: age 17, fare 52, male, first class, embarked at S.

In [ ]:
sex_code = {'male': 0, 'female': 1}
port_code = {'Q': 0, 'C': 1, 'S': 2}

passenger = {'age': 17, 'fare': 52, 'sex': 'male', 'pclass': 1, 'embarked': 'S'}
vec = np.array([passenger['age'], passenger['fare'], sex_code[passenger['sex']],
                passenger['pclass'], port_code[passenger['embarked']]])
print(vec)

## 4. Bag of words and a tiny recommender

In [ ]:
texts = {'A': 'hi how are you', 'B': 'my name is riya', 'C': 'this is 2023'}

cv = CountVectorizer()
bow = cv.fit_transform(texts.values()).toarray()

# one row per text, one column per vocabulary word
pd.DataFrame(bow, index=texts.keys(), columns=cv.get_feature_names_out())

In [ ]:
# Euclidean distance between every pair of texts
names = list(texts)
for i in range(3):
    for j in range(i + 1, 3):
        d = np.linalg.norm(bow[i] - bow[j])
        print(names[i], names[j], round(d, 2))

In [ ]:
def recommend(liked):
    """Return the other text whose vector is closest to the liked one."""
    i = names.index(liked)
    dists = {names[j]: np.linalg.norm(bow[i] - bow[j]) for j in range(len(names)) if j != i}
    return min(dists, key=dists.get)

print('liked B -> recommend', recommend('B'))

## 5. Row and column vectors

In [ ]:
x = np.array([5.1, 3.5, 1.4, 0.2])
print(x.shape)                 # (4,): no row/column form
print(x.reshape(1, -1).shape)  # (1, 4): row vector
print(x.reshape(-1, 1).shape)  # (4, 1): column vector

# one whole column, the species of all 150 flowers,
# is a 150 x 1 column vector
print(y.to_numpy().reshape(-1, 1).shape)